In [2]:
#importing 
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset as HFDataset
import evaluate
import pandas as pd
import numpy as np
import torch

c:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── Load processed data ───────────────────────────────────────
syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

In [4]:
#Model selection
MODEL_NAME = "xlm-roberta-base"
# MODEL_NAME = "bert-base-multilingual-cased"


In [5]:
# ── Detect GPU & set memory-efficient dtype ────────────────────
device    = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16  = torch.cuda.is_available()   # auto-enable on GPU
print(f"Using device : {device}")
print(f"FP16 enabled : {use_fp16}")
print(f"Model        : {MODEL_NAME}\n")

Using device : cuda
FP16 enabled : True
Model        : xlm-roberta-base



In [6]:
# ── Tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["review_text"],
        truncation=True,
        padding="max_length",
        max_length=128          # 128 saves memory vs 512; increase if needed
    )

In [8]:
# ── Dataset helper ─────────────────────────────────────────────
def to_hf_dataset(df):
    return HFDataset.from_dict({
        "review_text": df["review_text"].tolist(),
        "label":       df["label"].tolist(),
        "word_count":  df["word_count"].tolist(),
        "rating":      df["rating"].tolist(),
        "slang_count": df["slang_count"].tolist(),
    })

train_hf = to_hf_dataset(syn_train).map(tokenize, batched=True)
val_hf   = to_hf_dataset(syn_val).map(tokenize,   batched=True)
test_hf  = to_hf_dataset(syn_test).map(tokenize,  batched=True)

Map: 100%|██████████| 448/448 [00:00<00:00, 10936.21 examples/s]


In [9]:
# ── Model ──────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# ── Metrics ────────────────────────────────────────────────────
f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds          = np.argmax(logits, axis=1)
    return {
        "f1":       f1_metric.compute(
                        predictions=preds, references=labels,
                        average="weighted")["f1"],
        "accuracy": acc_metric.compute(
                        predictions=preds, references=labels)["accuracy"],
    }

c:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4102.16it/s]
XLMRobertaForSequenceClassification LOA

In [10]:
# ── Early stopping callback ────────────────────────────────────
# Stops training if F1 doesn't improve for 3 consecutive eval epochs
early_stop = EarlyStoppingCallback(
    early_stopping_patience  = 3,
    early_stopping_threshold = 0.001   # minimum delta to count as improvement
)